<a href="https://colab.research.google.com/github/19mddill/Deep_Learning_Notebooks/blob/main/training_DNN_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import tensorflow as tf
import numpy as np

In [2]:
dense = tf.keras.layers.Dense(50,activation="relu",kernel_initializer = "he_normal")

In [3]:
he_ag_init = tf.keras.initializers.VarianceScaling(scale=2, mode="fan_avg", distribution="uniform")

In [4]:
dense = tf.keras.layers.Dense(50,activation="sigmoid",kernel_initializer=he_ag_init)

In [5]:
leaky_relu = tf.keras.layers.LeakyReLU(alpha=0.2)
dense = tf.keras.layers.Dense(50,activation=leaky_relu,kernel_initializer="he_normal")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [6]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28,28]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(300,activation="relu",kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(100,activation="relu",kernel_initializer="he_normal"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(10,activation="softmax")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 784)            │         3,136 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 300)            │         1,200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 271,346 (1.04 MB)

 Trainable params: 268,978 (1.03 MB)

 Non-trainable params: 2,368 (9.25 KB)

In [8]:
[(var.name,var.trainable) for var in model.layers[1].variables]

[('gamma', True),
 ('beta', True),
 ('moving_mean', False),
 ('moving_variance', False)]

In [9]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28,28]),
    tf.keras.layers.Dense(300,kernel_initializer="he_normal",use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(100,kernel_initializer="he_normal",use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(10,activation="softmax")
])

# **Resuing Pretrained Layer**


## **Fetching the MNIST**

In [27]:
fashion_mnnist = tf.keras.datasets.fashion_mnist.load_data()
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [28]:
(X_train_full,y_train_full),(X_test,y_test) = fashion_mnnist

In [29]:
X_train_full.shape,X_test.shape

((60000, 28, 28), (10000, 28, 28))

In [30]:
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

In [31]:
X_valid.shape

(5000, 28, 28)

In [32]:
X_train, X_valid, X_test = X_train / 255., X_valid / 255., X_test / 255.

In [41]:
pos_class_id = class_names.index("Pullover")
neg_class_id = class_names.index("T-shirt/top")
pos_class_id,neg_class_id


(2, 0)

In [40]:
y_train[455]

np.uint8(3)

# **test**

In [50]:
y = np.array([1,2,1,4,1,6,1,2]) #Blessed

mask_for_two = (y % 2) == 0

(y[mask_for_two] == 2).astype(np.float32)


array([1., 0., 0., 1.], dtype=float32)

## **Defining a new data splitter**

In [52]:
from types import new_class
def split_dataset(X,y):
  y_for_B = (y == pos_class_id) | (y == neg_class_id)
  y_A = y[~y_for_B] # (y != pos_class_id) & (y != neg_class_id)
  y_B = (y[y_for_B] == pos_class_id).astype(np.float32)
  old_class_ids = list(set(range(10))-set([neg_class_id,pos_class_id]))
  for old_class_id,new_class_id in zip(old_class_ids,range(8)):
    y_A[y_A == old_class_id] = new_class_id
  return ((X[~y_for_B],y_A),(X[y_for_B],y_B))

## **Splitting the data**

In [53]:
(X_train_A, y_train_A), (X_train_B, y_train_B) = split_dataset(X_train, y_train)
(X_valid_A, y_valid_A), (X_valid_B, y_valid_B) = split_dataset(X_valid, y_valid)
(X_test_A, y_test_A), (X_test_B, y_test_B) = split_dataset(X_test, y_test)

In [54]:
X_train_B = X_train_B[:200]
y_train_B = y_train_B[:200]

## **Execution**

In [55]:
tf.random.set_seed(42)

In [58]:
model_A = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(8, activation="softmax")
])

model_A.compile(loss="sparse_categorical_crossentropy",
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
                metrics=["accuracy"])


In [59]:
history = model_A.fit(X_train_A, y_train_A, epochs=20,
                      validation_data=(X_valid_A, y_valid_A))

Epoch 1/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.6567 - loss: 1.1464 - val_accuracy: 0.7806 - val_loss: 0.6734
Epoch 2/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8160 - loss: 0.5786 - val_accuracy: 0.8413 - val_loss: 0.4932
Epoch 3/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8502 - loss: 0.4665 - val_accuracy: 0.8619 - val_loss: 0.4235
Epoch 4/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8658 - loss: 0.4123 - val_accuracy: 0.8696 - val_loss: 0.3857
Epoch 5/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8739 - loss: 0.3792 - val_accuracy: 0.8797 - val_loss: 0.3614
Epoch 6/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8805 - loss: 0.3563 - val_accuracy: 0.8842 - val_loss: 0.3447
Epoch 7/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8857 - loss: 0.3392 - val_accuracy: 0.8847 - val_loss: 0.3322
Epoch 8/20
1376/1376 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8895 - loss: 0.3258 - 

In [61]:
model_A.save("my_model_A.keras") #already done model by an expert.cant take all the layers of model_A.

In [62]:
tf.random.set_seed(42)

In [63]:
tf.random.set_seed(42)
model_B = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(100, activation="relu",
                          kernel_initializer="he_normal"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_B.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
                metrics=["accuracy"])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [64]:
history = model_B.fit(X_train_B, y_train_B, epochs=20,
                      validation_data=(X_valid_B, y_valid_B))

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.5550 - loss: 0.7318 - val_accuracy: 0.5173 - val_loss: 0.7293
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5600 - loss: 0.6919 - val_accuracy: 0.5242 - val_loss: 0.6944
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5650 - loss: 0.6616 - val_accuracy: 0.5430 - val_loss: 0.6674
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5850 - loss: 0.6378 - val_accuracy: 0.5796 - val_loss: 0.6456
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6100 - loss: 0.6182 - val_accuracy: 0.6142 - val_loss: 0.6273
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6650 - loss: 0.6016 - val_accuracy: 0.6766 - val_loss: 0.6113
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7350 - loss: 0.5867 - val_accuracy: 0.7260 - val_loss: 0.5968
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8050 - loss: 0.5729 - val_accuracy: 0.7913 - val_loss: 0.5835


In [65]:
model_B.evaluate(X_test_B, y_test_B)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9110 - loss: 0.4730


[0.4730096459388733, 0.9110000133514404]

## ***Pretraining***

In [66]:
model_A = tf.keras.models.load_model("my_model_A.keras")